In [12]:
# --- Step 1: df_testのロードとセグメンテーションの適用 ---
print("--- Step 1: df_testのロードとセグメンテーションの適用 ---")

import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model 

# --- パスの定義 ---
DATA_DIR = '../data/processed_test/'
MODEL_DIR = '../models/'
OUTPUT_DIR = '../submission/'

PROCESSED_FILE = 'test_features.parquet' # 修正済みファイル名
SEGMENTATION_FILE = 'df_segmentation_test.parquet' 

# --- 1. df_test (特徴量) のロード ---
try:
    # df_test という変数名で特徴量をロード
    df_test = pd.read_parquet(os.path.join(DATA_DIR, PROCESSED_FILE))
    
    # 予測結果を提出形式に合わせるため、IDカラムを保持
    test_ids = df_test['ID'] # 🚨 カラム名を 'ID' に修正 (ご提示のデータに合わせる)
    X_test_full = df_test.drop(columns=['ID']) # モデル入力用データ (IDなし)
    X_test_full.columns = X_test_full.columns.str.replace(':', '_') # Keras入力用(ご提示データで確認できないが念のため)
    print(f"✅ df_test (特徴量) を {len(df_test):,} 件ロードしました。")
    
except FileNotFoundError as e:
    print(f"❌ 特徴量ファイルが見つかりません: {e}")
    raise


# --- 2. BCD選別ルールとセグメントファイルの作成/更新 ---

df_segmentation_test = df_test[['ID']].copy()
df_segmentation_test.rename(columns={'ID': 'id'}, inplace=True) # 内部処理を 'id' に統一
df_segmentation_test['Segment_Type'] = 'None'

segment_required_cols = ['建物の構造', '都道府県名', '取引時点での築年数', '地区名_頻度_log']
transaction_cols = [col for col in df_test.columns if col.startswith('取引の事情等')]

for col in segment_required_cols + transaction_cols:
     if col in df_test.columns:
        df_segmentation_test[col] = df_test[col]

print("-" * 50)
print("--- BCDセグメントの選別適用 ---")

# B: 特殊ノイズのマスク定義 (取引の事情等)
shijo_cols_B = [col for col in transaction_cols if any(keyword in col for keyword in ['調停・競売等', '関係者間取引', '他の権利・負担付き', '瑕疵有りの可能性'])]
if shijo_cols_B:
    mask_B = df_segmentation_test[shijo_cols_B].any(axis=1)
    df_segmentation_test.loc[mask_B, 'Segment_Type'] = 'B:特殊ノイズ'
    print(f"   - Bセグメント選定: {mask_B.sum():,}件")
else:
    print("   - Bセグメント選定スキップ: 取引の事情等カラムが見つかりません。")

# C: 困難構造のマスク定義
mask_C = ((df_segmentation_test['建物の構造'] == '軽量鉄骨造') | 
          (df_segmentation_test['建物の構造'] == 'ブロック造') | 
          (df_segmentation_test['建物の構造'] == 'ＲＣ、木造'))
df_segmentation_test.loc[mask_C & (df_segmentation_test['Segment_Type'] == 'None'), 'Segment_Type'] = 'C:困難構造'
print(f"   - Cセグメント選定: {(df_segmentation_test['Segment_Type'] == 'C:困難構造').sum():,}件")

# D: プレミア/希少の選定
mask_D = (((df_segmentation_test['都道府県名'] == '東京都') & (df_segmentation_test['取引時点での築年数'] < 5)) | 
          (df_segmentation_test['地区名_頻度_log'].isnull()))
df_segmentation_test.loc[mask_D & (df_segmentation_test['Segment_Type'] == 'None'), 'Segment_Type'] = 'D:プレミア/希少'
print(f"   - Dセグメント選定: {(df_segmentation_test['Segment_Type'] == 'D:プレミア/希少').sum():,}件")

# A/E: 残り (モデル処理対象)
df_segmentation_test.loc[df_segmentation_test['Segment_Type'] == 'None', 'Segment_Type'] = 'A:高精度'
print(f"   - A/E対象件数 (A:高精度として一時統合): {(df_segmentation_test['Segment_Type'] == 'A:高精度').sum():,}件")

# セグメンテーション結果をファイルに保存 (次の推論ステップのために必須)
df_segmentation_test_final = df_segmentation_test[['id', 'Segment_Type']].copy()
os.makedirs(DATA_DIR, exist_ok=True)
df_segmentation_test_final.to_parquet(os.path.join(DATA_DIR, SEGMENTATION_FILE), index=False)
print(f"\n✅ df_segmentation_test.parquet を保存しました。")
print("-" * 50)

--- Step 1: df_testのロードとセグメンテーションの適用 ---
✅ df_test (特徴量) を 19,466 件ロードしました。
--------------------------------------------------
--- BCDセグメントの選別適用 ---
   - Bセグメント選定: 147件
   - Cセグメント選定: 1件
   - Dセグメント選定: 382件
   - A/E対象件数 (A:高精度として一時統合): 18,936件

✅ df_segmentation_test.parquet を保存しました。
--------------------------------------------------


In [13]:
df_test.columns

Index(['ID', '市区町村コード', '都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）',
       '面積（㎡）', '建築年', '建物の構造', '用途', '今後の利用目的', '都市計画', '建ぺい率（％）', '容積率（％）',
       '取引時点', '取引の事情等', '取引時点_年', '建築年_西暦', '築年数_欠損', '取引時点での築年数',
       '取引の事情等_調停・競売等', '取引の事情等_関係者間取引', '取引の事情等_その他', '改装_改装済', '改装_未改装',
       '間取り_grouped_３ＬＤＫ', '間取り_grouped_１Ｋ', '間取り_grouped_２ＬＤＫ',
       '間取り_grouped_４ＬＤＫ', '間取り_grouped_１ＬＤＫ', '間取り_grouped_２ＤＫ',
       '間取り_grouped_欠損値', '間取り_grouped_１ＤＫ', '間取り_grouped_３ＤＫ',
       '間取り_grouped_１Ｒ', '間取り_grouped_オープンフロア', '間取り_grouped_２ＬＤＫ＋Ｓ',
       '間取り_grouped_４ＤＫ', '間取り_grouped_２Ｋ', '間取り_grouped_その他', '市区町村人口密度',
       '犯罪発生率', '人口密度', '都市計画_高価格帯', '都市計画_中価格帯', '都市計画_低価格帯', '築年数_2乗',
       '築年数_3乗', '築年数_log', '面積_log', '面積_平方根', '築年数×面積', '築年数×駅距離',
       '築年数×建ぺい率', '築年数×容積率', '築年数×人口密度', '面積×駅距離', '面積×建ぺい率', '面積×容積率',
       '面積×人口密度', '建築可能性', '容積率_建ぺい率比', '建ぺい率_2乗', '容積率_2乗', '駅距離_逆数',
       '駅距離_log', '駅距離_2乗', '駅距離×建ぺい率', '駅距離×容積率', '人口密度_log', '市区町村人口密度_log',
  

In [14]:
# --- Step 2: モデルのロードと A/E 分離の実行 (二値分類モデル使用) ---
print("\n--- Step 2: モデルのロードと A/E 分離の実行 ---")

# TensorFlow/Kerasのロード
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model 
from sklearn.metrics import mean_absolute_error # 念のためインポート

# --- 訓練済み要素のロード ---
# 1. A/Dセグメント XGBoost モデル
model_a_new = joblib.load(os.path.join(MODEL_DIR, 'model_a_new.pkl'))
# 2. Eセグメント NN モデル (Embedding含む)
model_e_nn = load_model(os.path.join(MODEL_DIR, 'model_e_nn.keras'))
# 3. B/Cパラメータと特徴量リスト
prediction_params = joblib.load(os.path.join(MODEL_DIR, 'prediction_params.pkl'))
BC_AVERAGE_LOG = prediction_params['bc_average_log']
NUMERIC_FEATURES = prediction_params['numeric_features']
CATEGORICAL_FEATURES = prediction_params['categorical_features']

# 🚨 ノイズ分類モデルのロード (保存ファイル名 'classifier_noise.pkl' をロードし、変数名 'classifier_ae' に格納)
try:
    classifier_ae = joblib.load(os.path.join(MODEL_DIR, 'classifier_noise.pkl'))
    print("✅ ノイズ分類モデル (classifier_ae) をロードしました。")
except FileNotFoundError:
    print("❌ ノイズ分類モデル (classifier_noise.pkl) が見つかりません。A/E分離をスキップします。")
    # ここで処理を中断するか、全てAとして統合する

print(f"✅ モデルとパラメータをロード完了。")



--- Step 2: モデルのロードと A/E 分離の実行 ---
✅ ノイズ分類モデル (classifier_ae) をロードしました。
✅ モデルとパラメータをロード完了。


In [15]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19466 entries, 0 to 19465
Data columns (total 83 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   19466 non-null  int64  
 1   市区町村コード              19466 non-null  int64  
 2   都道府県名                19466 non-null  object 
 3   市区町村名                19466 non-null  object 
 4   地区名                  19463 non-null  object 
 5   最寄駅：名称               19453 non-null  object 
 6   最寄駅：距離（分）            18509 non-null  float64
 7   面積（㎡）                19466 non-null  int64  
 8   建築年                  18804 non-null  object 
 9   建物の構造                18201 non-null  object 
 10  用途                   13480 non-null  object 
 11  今後の利用目的              18439 non-null  object 
 12  都市計画                 19122 non-null  object 
 13  建ぺい率（％）              19045 non-null  float64
 14  容積率（％）               19045 non-null  float64
 15  取引時点                 19466 non-null 

In [19]:
# --- X_test_ae_candidate の reindex 前後の比較 ---
print("\n--- reindex 処理前後のカラム構成比較 ---")

# 1. ノイズ分類器ブースターから訓練時の正確な特徴量リストを再取得
# 🚨 前回の Step 2 の実行が成功しているため、このリストは存在します。
try:
    model_feature_names = classifier_ae.get_booster().feature_names
except AttributeError:
    print("❌ エラー: classifier_ae が見つからないか、訓練されていません。")
    # ここで処理を終了する場合は raise

# 2. A/E 候補のデータ抽出 (再実行)
mask_ae_candidate = (df_segmentation_test['Segment_Type'] == 'A:高精度').values
# 🚨 X_test_processed は Step 2 の最初で作成された前処理済みデータです
X_test_ae_candidate_BEFORE = X_test_processed.loc[mask_ae_candidate].copy()


# 3. ビフォー: reindex前のデータフレームのカラムリストと最初の1行
print(f"\n[Before] reindex前 (X_test_ae_candidate): {len(X_test_ae_candidate_BEFORE.columns)} カラム")
print("  --- カラム名リスト (先頭20, 末尾2) ---")
print(pd.Series(X_test_ae_candidate_BEFORE.columns).head(20).tolist() + ['...', '...'])
print(f"\n  --- データフレームの最初の1行のデータ型 ---")
# データフレームの最初の1行を表示 (値と型)
print(X_test_ae_candidate_BEFORE.head(1).T)


# 4. アフター: reindex後のデータフレームのカラムリストと最初の1行

# 訓練時の特徴量リストで再インデックス化
X_test_ae_candidate_AFTER = X_test_ae_candidate_BEFORE.reindex(columns=model_feature_names)

print(f"\n[After] reindex後 (X_test_ae_candidate_reindexed): {len(X_test_ae_candidate_AFTER.columns)} カラム")
print("  --- カラム名リスト (先頭20, 末尾2) ---")
print(pd.Series(X_test_ae_candidate_AFTER.columns).head(20).tolist() + ['...', '...'])
print(f"\n  --- データフレームの最初の1行のデータ型 ---")
# データフレームの最初の1行を表示 (値と型)
print(X_test_ae_candidate_AFTER.head(1).T)


--- reindex 処理前後のカラム構成比較 ---

[Before] reindex前 (X_test_ae_candidate): 74 カラム
  --- カラム名リスト (先頭20, 末尾2) ---
['最寄駅：距離（分）', '面積（㎡）', '建ぺい率（％）', '容積率（％）', '取引時点_年', '建築年_西暦', '築年数_欠損', '取引時点での築年数', '取引の事情等_調停・競売等', '取引の事情等_関係者間取引', '取引の事情等_その他', '改装_改装済', '間取り_grouped_オープンフロア', '間取り_grouped_欠損値', '間取り_grouped_１ＤＫ', '間取り_grouped_１Ｋ', '間取り_grouped_１ＬＤＫ', '間取り_grouped_１Ｒ', '間取り_grouped_２ＤＫ', '間取り_grouped_２Ｋ', '...', '...']

  --- データフレームの最初の1行のデータ型 ---
                    0
最寄駅：距離（分）        26.0
面積（㎡）              75
建ぺい率（％）          40.0
容積率（％）           60.0
取引時点_年         2020.0
...               ...
市区町村名          札幌市中央区
地区名               旭ケ丘
最寄駅：名称           円山公園
建物の構造              ＲＣ
取引時点       2020年第２四半期

[74 rows x 1 columns]

[After] reindex後 (X_test_ae_candidate_reindexed): 81 カラム
  --- カラム名リスト (先頭20, 末尾2) ---
['都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）', '面積（㎡）', '建物の構造', '建ぺい率（％）', '容積率（％）', '取引時点', '取引時点_年', '建築年_西暦', '築年数_欠損', '取引時点での築年数', '取引の事情等_その他事情有り', '取引の事情等_他の権利・負担付

In [18]:
# --- Step 2: A/E 分離の実行 (最終試行: ブースター特徴量リスト強制使用) ---
print("\n--- Step 2: A/E 分離の実行 (最終試行: ブースター特徴量リスト強制使用) ---")

# 4. A/E 候補のデータ抽出
mask_ae_candidate = (df_segmentation_test['Segment_Type'] == 'A:高精度').values
X_test_ae_candidate = X_test_processed.loc[mask_ae_candidate] # 前処理済みデータから抽出

if len(X_test_ae_candidate) == 0:
    print("    - ❌ A/E候補データが0件です。A/E分離をスキップします。")
else:
    # 🚨🚨🚨 最終エラー対策: classifier_aeが訓練時に使用した正確な特徴量リストを取得 🚨🚨🚨
    
    try:
        # モデルが訓練時に使用した正確な特徴量名のリストをブースターから直接取得
        model_feature_names = classifier_ae.get_booster().feature_names
        print(f"    - ✅ ノイズ分類器ブースターから特徴量リストを {len(model_feature_names)}件 取得。")

    except AttributeError:
        # 稀なケースとしてブースターから取得できない場合 (通常は発生しない)
        print("❌ ノイズ分類器ブースターから特徴量リストを取得できませんでした。")
        raise
    
    # X_test_ae_candidate のカラムを、モデルの feature_names に合わせて再インデックス化
    # 🚨 カテゴリカル変数のdtypeが維持されるように、fill_valueは指定しない（NaNのままにする）
    X_test_ae_candidate_reindexed = X_test_ae_candidate.reindex(columns=model_feature_names)
    
    # reindex後にNaNになる数値型の欠損値を、XGBoostのmissingパラメータで処理させるため、そのまま渡す
    
    # カテゴリカル変数の dtype を強制的に 'category' に戻す (reindexで変わることがあるため)
    categorical_cols = prediction_params['categorical_features']
    for col in categorical_cols:
        if col in X_test_ae_candidate_reindexed.columns:
            # 訓練時と同じカテゴリカル型に強制変換
            X_test_ae_candidate_reindexed[col] = X_test_ae_candidate_reindexed[col].astype('category') 
        
    print("    - ✅ ブースターの特徴量リストで X_test_ae_candidate を再インデックス化完了。")

    # 5. ノイズ分類モデルで E のクラスを直接予測 (predict()版)
    # 🚨 再インデックス化したデータフレームを使用
    y_pred_class_test = classifier_ae.predict(X_test_ae_candidate_reindexed)
    
    # 6. 予測が 'E' (1) であるデータのインデックスを抽出
    e_indices_test = X_test_ae_candidate_reindexed[y_pred_class_test == 1].index

    # 7. df_segmentation_test の Eセグメントのラベルを更新
    df_segmentation_test.loc[e_indices_test, 'Segment_Type'] = 'E:高誤差ノイズ'
    
    count_e_test = (df_segmentation_test['Segment_Type'] == 'E:高誤差ノイズ').sum()
    count_a_test = (df_segmentation_test['Segment_Type'] == 'A:高精度').sum()

    print(f"    - ✅ A/E 分離完了。ノイズ分類器のクラス予測を適用。")
    print(f"    - 💡 最終的な A セグメント件数: {count_a_test:,}件")
    print(f"    - 💡 最終的な E セグメント件数: {count_e_test:,}件") 

print("-" * 50)


--- Step 2: A/E 分離の実行 (最終試行: ブースター特徴量リスト強制使用) ---
    - ✅ ノイズ分類器ブースターから特徴量リストを 81件 取得。
    - ✅ ブースターの特徴量リストで X_test_ae_candidate を再インデックス化完了。
    - ✅ A/E 分離完了。ノイズ分類器のクラス予測を適用。
    - 💡 最終的な A セグメント件数: 18,931件
    - 💡 最終的な E セグメント件数: 5件
--------------------------------------------------


In [24]:
import numpy as np
import pandas as pd
import os

# --------------------------------------------------------------------------
# --- Step 3: 各セグメントモデルによる予測の実行と統合 ---
# --------------------------------------------------------------------------
print("\n--- Step 3: 各セグメントモデルによる予測の実行と統合 ---")

# 最終予測結果を格納する Series を初期化 (df_testのID順)
y_pred_test = pd.Series(index=df_test.index, dtype=float)

# 1. B/C セグメントの予測 (BC_AVERAGE_LOGを適用)
mask_bc = (df_segmentation_test['Segment_Type'] == 'B:特殊ノイズ') | \
          (df_segmentation_test['Segment_Type'] == 'C:困難構造')

BC_AVERAGE_LOG = prediction_params['bc_average_log'] 
y_pred_test.loc[mask_bc] = BC_AVERAGE_LOG
print(f"    - ✅ B/C セグメント予測完了。({mask_bc.sum():,}件, 予測値(log): {BC_AVERAGE_LOG:.6f})")


# 2. A/D セグメントの予測 (model_a_new を適用) - (前回成功済み)
mask_ad = (df_segmentation_test['Segment_Type'] == 'D:プレミア/希少') | \
          (df_segmentation_test['Segment_Type'] == 'A:高精度')
X_test_ad = X_test_processed.loc[mask_ad.values]

if len(X_test_ad) > 0:
    # A/Dモデルの予測に必要な特徴量リストを取得
    try:
        ad_model_feature_names = model_a_new.get_booster().feature_names
        # print(f"    - ✅ A/Dモデルブースターから特徴量リストを {len(ad_model_feature_names)}件 取得。")

    except AttributeError:
        # A/Dモデルの読み込みエラー
        ad_model_feature_names = prediction_params['numeric_features'] + prediction_params['categorical_features']
    
    X_test_ad_reindexed = X_test_ad.reindex(columns=ad_model_feature_names)
    
    # カテゴリカル型の維持
    categorical_cols = prediction_params['categorical_features']
    for col in categorical_cols:
        if col in X_test_ad_reindexed.columns:
            X_test_ad_reindexed[col] = X_test_ad_reindexed[col].astype('category') 

    y_pred_test.loc[mask_ad] = model_a_new.predict(X_test_ad_reindexed)
    print(f"    - ✅ A/D セグメント予測完了。({mask_ad.sum():,}件)")
else:
    print("    - ⚠️ A/D セグメントの件数が0のため、予測をスキップしました。")


# 3. E セグメントの予測 (model_e_nn を適用)
mask_e = df_segmentation_test['Segment_Type'] == 'E:高誤差ノイズ'
X_test_e = X_test_processed.loc[mask_e.values]

if len(X_test_e) > 0:
    # 🚨🚨🚨 修正箇所 1: Eモデルの予測直前に reindex 処理を追加 🚨🚨🚨
    
    # Eモデルも A/D モデルと同じ特徴量セットを期待すると仮定し、reindexを実行
    # A/Dモデルのブースターから取得したリスト (ad_model_feature_names) を使用
    X_test_e_reindexed = X_test_e.reindex(columns=ad_model_feature_names)
    
    # 🚨🚨🚨 修正箇所 2: reindex後のデータフレームから辞書を作成 🚨🚨🚨
    X_test_e_dict = {}
    
    for name in prediction_params['numeric_features']:
        # reindex後のデータフレームから抽出
        if name in X_test_e_reindexed.columns:
            # 訓練時と同様に NaN は 0 で埋める（Keras予測では必須）
            X_test_e_dict[name] = np.array(X_test_e_reindexed[name].fillna(0)) 
            
    for name in prediction_params['categorical_features']:
        # reindex後のデータフレームから抽出
        if name in X_test_e_reindexed.columns:
            col = X_test_e_reindexed[name].copy()
            # Keras Embedding Layer の入力として category 型を object に戻す
            col = col.astype(object) 
            X_test_e_dict[name] = col.fillna('missing').values # 訓練時と同様に欠損値を 'missing' で埋める
        
    y_pred_test_e_array = model_e_nn.predict(X_test_e_dict, verbose=0).flatten()
    y_pred_test.loc[mask_e] = y_pred_test_e_array
    print(f"    - ✅ E セグメント予測完了。({len(X_test_e):,}件)")
else:
    print(f"    - ⚠️ E セグメントの件数が0のため、予測をスキップしました。")


# 4. 最終結果の統合と保存
y_pred_test.name = '取引価格（総額）_log'
total_predictions = y_pred_test.notna().sum()
print(f"\n✅ 全セグメントの予測結果を統合完了。総予測件数: {total_predictions:,}件")

# 最終提出ファイルを作成 (IDと予測結果)
if 'test_ids' not in locals():
    test_ids = df_test['ID'].values 

df_submission = pd.DataFrame({
    'ID': test_ids,
    '取引価格（総額）_log': y_pred_test.values
})

OUTPUT_DIR = '../output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_submission.to_csv(os.path.join(OUTPUT_DIR, 'submission.csv'), index=False)
print(f"✅ 最終予測結果 (submission.csv) を {os.path.join(OUTPUT_DIR, 'submission.csv')} に保存しました。")
print("-" * 50)


--- Step 3: 各セグメントモデルによる予測の実行と統合 ---
    - ✅ B/C セグメント予測完了。(148件, 予測値(log): 7.238565)
    - ✅ A/D セグメント予測完了。(19,313件)
    - ✅ E セグメント予測完了。(5件)

✅ 全セグメントの予測結果を統合完了。総予測件数: 19,466件
✅ 最終予測結果 (submission.csv) を ../output/submission.csv に保存しました。
--------------------------------------------------


In [25]:
df_submission

,ID,取引価格（総額）_log
0,1000000,6.873213
1,1000056,7.311492
2,1000108,6.367496
3,1000109,6.888304
4,1000110,6.354736
...,...,...
19461,47003828,7.475312
19462,47003829,7.249241
19463,47003880,7.186438
19464,47006648,7.303598
